In [15]:
from typing import TypedDict
import importlib
from langgraph.graph import StateGraph, END,START
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import notes_ai
notes_ai = importlib.reload(notes_ai)
chercher_notes = notes_ai.chercher_notes
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver  


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3331.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
from langgraph.graph import StateGraph, END,START

# 1. Define state
from typing import List, TypedDict


class State(TypedDict):
    user_query: str
    search_queries: List[str]
    notes: List[str]
    answer: str

In [17]:
llm = ChatGroq(
    model="llama-3.1-8b-versatile" ,
      # fast + good for testing
)
checkpointer = InMemorySaver()


In [18]:
def planner_node(state):
    question = state["user_query"]

    prompt = f"""
You are a search planner.

Convert the user question short search queries.

Rules:
- Output ONLY a Python list of strings
- No explanation
- No numbering
- Queries must be short and keyword-based

User question:
{question}
"""

    response = llm.invoke(prompt)

    # convert string output → python list safely
    queries = eval(response.content)

    return {
        "search_queries": queries
    }

In [19]:
def retriever_node(state):
    all_notes = []

    for q in state["search_queries"]:
        notes = chercher_notes(q)
        all_notes.extend(notes)

    # remove duplicates
    all_notes = list(set(all_notes))

    return {
        "notes": all_notes
    }

In [ ]:
def answer_node(state):
    question = state["user_query"]
    notes = state["notes"]

    context = "\n".join([f"- {n}" for n in notes])

    prompt = f"""
You are a second brain assistant.

You MUST answer ONLY using the provided notes.
If the notes do not contain the answer, say: "I don't have enough information in my notes."

---

NOTES:

{context}

---

QUESTION:
{question}

---

Answer clearly and concisely:
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [ ]:
builder = StateGraph(State)
builder.add_node("planner", planner_node)
builder.add_node("retriever", retriever_node)
builder.add_node("answer", answer_node)
builder.set_entry_point("planner")
builder.add_edge(START, "planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "answer")
builder.add_edge("answer", END)
graph = builder.compile(checkpointer=checkpointer)


In [22]:
config = {"configurable": {"thread_id": "1"}}
result = graph.invoke(
    {"user_query": "Explain AI in 1 sentence"}, 
    config=config
)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
print(result)

{'user_query': 'Explain AI in 1 sentence', 'search_queries': ['What is AI', 'Artificial Intelligence definition'], 'notes': ["Quote - Naval Ravikant: 'Play iterated games. All the returns in life come from compound interest.'", 'Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.', '1-on-1: Improve communication, lead backend project.', 'Japanese basics: Arigatou = thank you, Sumimasen = sorry/excuse me.'], 'answer': "I don't have enough information in my notes."}


In [23]:
from langsmith import Client

client = Client()
dataset_name = "Notes_App_Test_Dataset"

# Create a new dataset in LangSmith
datasets = client.list_datasets()

dataset = next((d for d in datasets if d.name == dataset_name), None)

# Add some test examples (Input = user_query, Output = expected_answer)
examples =[
(
        {"user_query": "How do I undo my last git commit without losing changes?"}, 
        {"expected_answer": "You can use the command 'git reset --soft HEAD~1'."}
    ),
    (
        {"user_query": "What should I get my Mom for her birthday?"}, 
        {"expected_answer": "You should get her a Kindle Paperwhite or a new set of ceramic gardening pots."}
    ),
    (
        {"user_query": "What is the ratio and temperature for a V60 pour over?"}, 
        {"expected_answer": "Use 15g of coffee and 250ml of water at 93°C."}
    ),
    (
        {"user_query": "What did I discuss with my manager in our 1-on-1?"}, 
        {"expected_answer": "You discussed your promotion timeline, improving cross-functional communication, and leading a major backend project this quarter."}
    )
]

for input_dict, output_dict in examples:
    client.create_example(
        inputs=input_dict,
        outputs=output_dict,
        dataset_id=dataset.id,
    )
print(f"Dataset '{dataset_name}' created successfully!")

Dataset 'Notes_App_Test_Dataset' created successfully!


In [24]:
def predict_answer(inputs: dict):
    # Extract the question from the dataset input
    question = inputs["user_query"]
    
    # Run your LangGraph
    config = {"configurable": {"thread_id": "eval_thread"}}
    result = graph.invoke({"user_query": question}, config=config)
    
    # Return ONLY the final answer for evaluation
    return {"answer": result["answer"]}

In [ ]:
def correctness_evaluator(run, example):
    # Get the expected answer from the dataset
    expected = example.outputs["expected_answer"]
    # Get the actual answer from your graph
    actual = run.outputs["answer"]
    
    # Ask Groq to grade it, allowing it to think first
    prompt = f"""
    You are an expert grader evaluating an AI assistant.
    
    Expected Answer: {expected}
    Actual Answer: {actual}
    
    Task:
    1. Briefly explain if the Actual Answer contains the core factual information from the Expected Answer.
    2. It is okay if the Actual Answer provides extra context, as long as the core facts match.
    3. End your response with exactly: [GRADE: YES] or [GRADE: NO]
    """
    
    response = llm.invoke(prompt).content.strip().upper()
    
    # Safely parse the grade
    if "[GRADE: YES]" in response:
        score = 1
    elif "[GRADE: NO]" in response:
        score = 0
    else:
        # Fallback just in case the LLM formats it weirdly
        score = 0 
        
    return {
        "key": "correctness", 
        "score": score, 
        "comment": response # Optional: LangSmith will save the LLM's explanation!
    }

In [ ]:
from langsmith.evaluation import evaluate

experiment_results = evaluate(
    predict_answer, # Your LangGraph wrapper
    data=dataset_name, # The dataset we created
    evaluators=[correctness_evaluator], # The grading function
    experiment_prefix="groq-llama-3-test", # Name of this test run
    client=client
)

View the evaluation results for experiment: 'groq-llama-3-test-6fad7faa' at:
https://eu.smith.langchain.com/o/3a5ee616-d418-413e-bb48-8d292f03604d/datasets/0b919c23-aa11-40f3-a9ff-cd0aae1f29b0/compare?selectedSessions=e11d6412-4b27-486e-8383-387ff7a955c9




0it [00:00, ?it/s]


🔍 Recherche : 'Manager 1-on-1 discussion'
  → 1-on-1: Improve communication, lead backend project.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.
  → Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.

🔍 Recherche : '1-on-1 meeting notes'
  → 1-on-1: Improve communication, lead backend project.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.
  → Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.

🔍 Recherche : 'Manager discussion topics'
  → 1-on-1: Improve communication, lead backend project.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive 

1it [00:01,  1.08s/it]


🔍 Recherche : 'V60 pour over ratio'
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.
  → Finance - Monthly Rules: Keep groceries under 400 TND. Invest 20% monthly.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.

🔍 Recherche : 'V60 pour over temperature'
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.
  → Recipe - Perfect Pizza Dough: 500g flour, 325ml warm water, 10g salt, 3g yeast. Ferment 48 hours.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.

🔍 Recherche : 'V60 brewing ratio'
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.
  → Finance - Monthly Rules: Keep groceries under 400 TND. Invest 20% monthly.
  → Recipe - Perfect Pizza Dough: 500g flour, 325ml warm water, 10g salt, 3g yeast. Ferment 48 hours.

🔍 Recherche : 'V60 brewing te

...

8it [00:06,  1.28it/s]


🔍 Recherche : 'Groq CEO'
  → 1-on-1: Improve communication, lead backend project.
  → Home Maintenance: Change AC filter every 3 months.
  → Finance - Monthly Rules: Keep groceries under 400 TND. Invest 20% monthly.

🔍 Recherche : 'Groq leadership'
  → 1-on-1: Improve communication, lead backend project.
  → Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.

🔍 Recherche : 'Groq founder'
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.
  → Quote - Naval Ravikant: 'Play iterated games. All the returns in life come from compound interest.'
  → Home Maintenance: Change AC filter every 3 months.

🔍 Recherche : 'CEO of Groq'
  → 1-on-1: Improve communication, lead backend project.
  → V60 Coffee: 15g coffee, 250ml water, 2:30 brew time.
  → Home Maintenance: Change AC filter every 3 months.


9it [00:07,  1.31it/s]


🔍 Recherche : 'What is AI'
  → Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.
  → Japanese basics: Arigatou = thank you, Sumimasen = sorry/excuse me.
  → 1-on-1: Improve communication, lead backend project.

🔍 Recherche : 'AI definition'
  → Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.
  → Japanese basics: Arigatou = thank you, Sumimasen = sorry/excuse me.
  → Quote - Naval Ravikant: 'Play iterated games. All the returns in life come from compound interest.'

🔍 Recherche : 'Artificial Intelligence'
  → Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.
  → 1-on-1: Improve communication, lead backend project.
  → Quote - Naval Ravikant: 'Play iterated games. All the returns in l

10it [00:08,  1.15it/s]


In [ ]:
result = graph.invoke(
    {"user_query": "What did I discuss with my manager in our 1-on-1?"}, 
    config=config
)


🔍 Recherche : '1-on-1 meeting notes'
  → 1-on-1: Improve communication, lead backend project.
  → Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.
  → Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.

🔍 Recherche : 'manager discussion history'
  → 1-on-1: Improve communication, lead backend project.
  → Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.
  → Watchlist: Dune Part Two, Severance.

🔍 Recherche : 'meeting summary'
  → 1-on-1: Improve communication, lead backend project.
  → Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.
  → Watchlist: Dune Part Two, Severance.


In [ ]:
print(result)

{'user_query': 'What did I discuss with my manager in our 1-on-1?', 'search_queries': ['1-on-1 meeting notes', 'manager discussion history', 'meeting summary'], 'notes': ['1-on-1: Improve communication, lead backend project.', 'Watchlist: Dune Part Two, Severance.', 'Project Idea - AI Assistant: Build a local AI assistant using LangGraph and Llama 3 that can search through my local markdown files and summarize them.', 'Book Summary - Atomic Habits: The core idea is that getting 1% better every day leads to massive compounding results over time. Focus on systems, not goals.', 'Meeting Notes - Q3 Planning: The main goals for Q3 are to increase user retention by 15% and successfully launch the new iOS mobile app.'], 'answer': 'Improve communication and lead the backend project.'}
